In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight

# --- Parâmetros de Configuração e Reprodutibilidade ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

IMG_HEIGHT = 33      # Número de bins de frequência
IMG_WIDTH = 81       # Número de bins de tempo
NUM_CLASSES = 3      # 'rest', 'left', 'right'
BATCH_SIZE = 32      # Um batch size de 32 é um bom padrão
EPOCHS = 80         # Número máximo de épocas (EarlyStopping vai parar antes)

# Mapeamento de rótulos para inteiros
label_map = {'rest': 0, 'left': 1, 'right': 2}

# --- Função de Carregamento (Sem alterações) ---
def load_spectrogram_data(filepath_c3, filepath_c4, label):
    spectrogram_c3 = np.load(filepath_c3.numpy()).astype(np.float32)
    spectrogram_c4 = np.load(filepath_c4.numpy()).astype(np.float32)

    min_c3, max_c3 = np.min(spectrogram_c3), np.max(spectrogram_c3)
    spectrogram_c3 = (spectrogram_c3 - min_c3) / (max_c3 - min_c3 + 1e-8) if (max_c3 - min_c3) > 1e-8 else spectrogram_c3

    min_c4, max_c4 = np.min(spectrogram_c4), np.max(spectrogram_c4)
    spectrogram_c4 = (spectrogram_c4 - min_c4) / (max_c4 - min_c4 + 1e-8) if (max_c4 - min_c4) > 1e-8 else spectrogram_c4

    combined_spectrogram = np.stack([spectrogram_c3, spectrogram_c4], axis=-1)
    
    return combined_spectrogram, label

# --- Criação do Dataset (Totalmente Reformulada) ---
def create_datasets(data_root_dir):
    all_filepaths_c3, all_filepaths_c4, all_labels = [], [], []

    print(f"Lendo arquivos de: {data_root_dir}")
    if not os.path.exists(data_root_dir):
        print(f"Erro: O diretório raiz não existe: {data_root_dir}")
        return None, None, None

    for class_name, label_idx in label_map.items():
        class_dir = os.path.join(data_root_dir, class_name)
        if not os.path.exists(class_dir):
            print(f"Aviso: Diretório da classe '{class_name}' não encontrado. Pulando.")
            continue
        
        for fname in os.listdir(class_dir):
            if fname.endswith('_C3.npy'):
                fp_c3 = os.path.join(class_dir, fname)
                fp_c4 = fp_c3.replace('_C3.npy', '_C4.npy')
                if os.path.exists(fp_c4):
                    all_filepaths_c3.append(fp_c3)
                    all_filepaths_c4.append(fp_c4)
                    all_labels.append(label_idx)

    if not all_filepaths_c3:
        print("Erro: Nenhum par de espectrogramas C3/C4 foi encontrado.")
        return None, None, None

    # 1. Divisão Estratificada dos Dados (A Correção Mais Importante)
    # Divide os caminhos dos arquivos em treino e validação antes de criar os datasets
    print("Dividindo dados em conjuntos de treinamento e validação...")
    train_c3, val_c3, train_c4, val_c4, train_labels, val_labels = train_test_split(
        all_filepaths_c3, all_filepaths_c4, all_labels,
        test_size=0.20,      # 20% dos dados para validação
        random_state=SEED,   # Garante a mesma divisão sempre
        stratify=all_labels  # Mantém a proporção de classes em ambos os conjuntos
    )

    # 2. Criação dos Datasets de Treino e Validação Separadamente
    def create_tf_dataset(files_c3, files_c4, labels):
        dataset = tf.data.Dataset.from_tensor_slices(((files_c3, files_c4), labels))
        # Mapeia a função de carregamento
        dataset = dataset.map(lambda x, y: tf.py_function(load_spectrogram_data, [x[0], x[1], y], (tf.float32, tf.int32)),
                              num_parallel_calls=tf.data.AUTOTUNE)
        # Garante a forma dos tensores
        dataset = dataset.map(lambda spec, label: (tf.ensure_shape(spec, (IMG_HEIGHT, IMG_WIDTH, 2)), tf.ensure_shape(label, ())),
                              num_parallel_calls=tf.data.AUTOTUNE)
        return dataset

    train_ds = create_tf_dataset(train_c3, train_c4, train_labels)
    val_ds = create_tf_dataset(val_c3, val_c4, val_labels)

    # 3. Otimização dos Pipelines
    # Embaralha, agrupa em lotes e pré-busca os dados de treino
    train_ds = train_ds.shuffle(buffer_size=len(train_labels)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    # Apenas agrupa e pré-busca os dados de validação (não precisa embaralhar)
    val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    print(f"Dataset criado: {len(train_labels)} amostras de treino, {len(val_labels)} amostras de validação.")
    
    # Retorna os rótulos de treino para o cálculo de pesos
    return train_ds, val_ds, train_labels

# --- Definição do Modelo CNN (com regularização) ---
def create_cnn_model_regularized(input_shape=(IMG_HEIGHT, IMG_WIDTH, 2), num_classes=NUM_CLASSES):
    model = models.Sequential(name="EEG_CNN_Regularized")
    l2_rate = 1e-4  # Taxa para a regularização L2

    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape, kernel_regularizer=regularizers.l2(l2_rate)))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.BatchNormalization()) # Ajuda a estabilizar o treinamento

    model.add(layers.Conv2D(64, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_rate)))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.BatchNormalization())

    model.add(layers.Conv2D(128, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_rate)))
    model.add(layers.MaxPooling2D((2, 2)))
    
    model.add(layers.Flatten())
    
    model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(l2_rate)))
    model.add(layers.Dropout(0.5)) # Dropout é uma forma de regularização
    
    model.add(layers.Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# --- Treinamento ---
if __name__ == '__main__':
    current_window_ms = 200
    DATA_ROOT_DIR = f'unified_spectrograms_{current_window_ms}ms' 

    print(f"\n--- Preparando datasets de {DATA_ROOT_DIR} ---")
    train_dataset, val_dataset, train_labels = create_datasets(DATA_ROOT_DIR)

    if train_dataset is None:
        print("Treinamento abortado devido a erro na criação do dataset.")
    else:
        # 4. Cálculo dos Pesos de Classe para combater desbalanceamento
        print("\n--- Calculando pesos de classe ---")
        class_weights_array = class_weight.compute_class_weight(
            'balanced',
            classes=np.unique(train_labels),
            y=train_labels
        )
        class_weights = dict(enumerate(class_weights_array))
        print(f"Pesos calculados: {class_weights}")

        # 5. Definindo Callbacks (Early Stopping)
        early_stopping = EarlyStopping(
            monitor='val_loss',      # Monitora a perda na validação
            patience=15,             # Para por 15 épocas se não houver melhora
            verbose=1,
            restore_best_weights=True # Salva e restaura os pesos da melhor época
        )

        model = create_cnn_model_regularized()
        model.summary()

        print("\n--- Iniciando Treinamento ---")
        history = model.fit(
            train_dataset,
            epochs=EPOCHS,
            validation_data=val_dataset,
            class_weight=class_weights, # Usa os pesos para balancear as classes
            callbacks=[early_stopping]  # Adiciona o Early Stopping
        )

        # --- Avaliação e Visualização ---
        print("\n--- Avaliação Final do Modelo (no conjunto de validação) ---")
        loss, accuracy = model.evaluate(val_dataset)
        print(f"Acurácia final no conjunto de validação: {accuracy*100:.2f}%")

        # Plotar histórico
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.plot(history.history['accuracy'], label='Acurácia de Treino')
        plt.plot(history.history['val_accuracy'], label='Acurácia de Validação')
        plt.legend()
        plt.title('Acurácia ao Longo das Épocas')
        plt.subplot(1, 2, 2)
        plt.plot(history.history['loss'], label='Perda de Treino')
        plt.plot(history.history['val_loss'], label='Perda de Validação')
        plt.legend()
        plt.title('Perda ao Longo das Épocas')
        plt.tight_layout()
        plt.show()

        # Salvar o modelo final (que é o melhor modelo graças a restore_best_weights=True)
        model.save(f'eeg_cnn_model_regularized_{current_window_ms}ms.keras')
        print(f"Modelo salvo como 'eeg_cnn_model_regularized_{current_window_ms}ms.keras'")


--- Preparando datasets de unified_spectrograms_200ms ---
Lendo arquivos de: unified_spectrograms_200ms
Dividindo dados em conjuntos de treinamento e validação...
Dataset criado: 6120 amostras de treino, 1530 amostras de validação.

--- Calculando pesos de classe ---
Pesos calculados: {0: np.float64(0.6666666666666666), 1: np.float64(1.3203883495145632), 2: np.float64(1.3465346534653466)}


c:\Codes\EEG_Study\venv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "EEG_CNN_Regularized"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 31, 79, 32)     │           608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 15, 39, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 15, 39, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 13, 37, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 6, 18, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 6, 18, 64)      │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 4, 16, 128)     │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 2, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 356,003 (1.36 MB)

 Trainable params: 355,811 (1.36 MB)

 Non-trainable params: 192 (768.00 B)


--- Iniciando Treinamento ---
Epoch 1/80
192/192 ━━━━━━━━━━━━━━━━━━━━ 14s 50ms/step - accuracy: 0.3298 - loss: 1.2924 - val_accuracy: 0.2405 - val_loss: 1.1314
Epoch 2/80
192/192 ━━━━━━━━━━━━━━━━━━━━ 13s 48ms/step - accuracy: 0.2530 - loss: 1.1323 - val_accuracy: 0.2634 - val_loss: 1.1295
Epoch 3/80
192/192 ━━━━━━━━━━━━━━━━━━━━ 14s 54ms/step - accuracy: 0.3130 - loss: 1.1274 - val_accuracy: 0.3810 - val_loss: 1.1225
Epoch 4/80
192/192 ━━━━━━━━━━━━━━━━━━━━ 14s 50ms/step - accuracy: 0.4510 - loss: 1.1200 - val_accuracy: 0.4529 - val_loss: 1.1239
Epoch 5/80
192/192 ━━━━━━━━━━━━━━━━━━━━ 14s 52ms/step - accuracy: 0.4511 - loss: 1.1236 - val_accuracy: 0.4993 - val_loss: 1.1172
Epoch 6/80
192/192 ━━━━━━━━━━━━━━━━━━━━ 15s 55ms/step - accuracy: 0.4777 - loss: 1.1074 - val_accuracy: 0.3922 - val_loss: 1.1234
Epoch 7/80
190/192 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.4974 - loss: 1.1236

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import butter, filtfilt
import mne # Substitui pyedflib
from pathlib import Path
import os

# --- Funções de Processamento de Dados (sem alterações) ---

def bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

def notch_filter(data, notch_freq, fs, quality=30):
    nyquist = 0.5 * fs
    freq = notch_freq / nyquist
    b, a = butter(2, [freq - 0.5 / quality, freq + 0.5 / quality], btype='bandstop')
    return filtfilt(b, a, data)

def compute_individual_epoch_stft(data, events, event_id, tmin, tmax, fs, nperseg, noverlap, nfft):
    """
    Compute STFT for each individual epoch of a specific event type.
    Retorna uma lista de dicionários, cada um contendo (f, t, Zxx) para uma época.
    """
    event_samples = [e[0] for e in events if e[2] == event_id]
    
    individual_stft_data = []
    
    for i, sample in enumerate(event_samples):
        start = sample + int(tmin * fs)
        end = sample + int(tmax * fs)
        epoch = data[start:end]
        
        if len(epoch) < nperseg:
            continue

        f, t, Zxx = signal.stft(epoch, fs=fs, window='hann', nperseg=nperseg, noverlap=noverlap, nfft=nfft)
        individual_stft_data.append({'epoch_idx': i, 'frequencies': f, 'times': t, 'magnitude': np.abs(Zxx)})
        
    return individual_stft_data

# --- Configurações Globais ---
pasta_raiz_sujeitos = Path('data/phisionet_data')
start_subject = 1
end_subject = 85
registros = ['R04', 'R08', 'R12']
# Mapeamento de eventos para o MNE e para nomes de pastas
event_map_mne = {'T0': 1, 'T1': 2, 'T2': 3} 
event_map_folders = {1: 'rest', 2: 'left', 3: 'right'}
epoch_duration_sec = 4
window_durations_ms = [200]

print(f"Iniciando a geração de spectrograms para sujeitos S{start_subject:03d} a S{end_subject:03d}")
print(f"e para as janelas: {window_durations_ms}ms")

# --- Loop Principal para Processamento (Aninhado) ---

for current_window_ms in window_durations_ms:
    current_window_s = current_window_ms / 1000.0
    
    print(f"\n--- Processando com Janela de {current_window_ms}ms ---")

    base_output_data_dir = f"unified_spectrograms_{current_window_ms}ms"
    for class_folder_name in event_map_folders.values():
        os.makedirs(os.path.join(base_output_data_dir, class_folder_name), exist_ok=True)
    print(f"Diretório de saída para o dataset unificado: {base_output_data_dir}")

    for subject_num in range(start_subject, end_subject + 1):
        subject_id = f'S{subject_num:03d}'
        subject_folder_path = pasta_raiz_sujeitos / subject_id

        if not subject_folder_path.is_dir():
            print(f"  Pasta do sujeito {subject_id} não encontrada. Pulando.")
            continue

        print(f"  Processando Sujeito: {subject_id}")

        for registro in registros:
            edf_file = next(subject_folder_path.glob(f'*{registro}.edf'), None)
            if not edf_file:
                print(f"    Arquivo {registro}.edf não encontrado. Pulando.")
                continue

            try:
                # --- NOVO: Carregar dados com MNE-Python ---
                # verbose='ERROR' para suprimir saídas excessivas do MNE
                raw = mne.io.read_raw_edf(edf_file, preload=True, verbose='ERROR')
                
                # Selecionar apenas os canais de interesse. MNE lida com os nomes exatos.
                # O MNE levanta um erro se os canais não existirem, o que é mais seguro.
                raw.pick_channels(['C3..', 'C4..'])
                
                # Obter a taxa de amostragem do arquivo
                fs = int(raw.info['sfreq'])

                # --- NOVO: Extrair eventos das anotações usando MNE ---
                # Isso substitui a leitura manual de anotações e a conversão
                events, _ = mne.events_from_annotations(raw, event_id=event_map_mne, verbose='ERROR')

                # Obter os dados dos sinais como um array NumPy
                # O formato é (n_channels, n_samples)
                signals_data = raw.get_data()
                c3_signal = signals_data[0]
                c4_signal = signals_data[1]

            except Exception as e:
                print(f"    ERRO ao processar {edf_file.name} com MNE: {e}. Pulando este arquivo.")
                continue

            # Recalcular parâmetros da STFT com a taxa de amostragem correta do arquivo
            nperseg = int(fs * current_window_s)
            noverlap = int(nperseg * 0.75)
            nfft = nperseg * 2
            nperseg = max(1, nperseg)
            noverlap = min(noverlap, nperseg - 1) if nperseg > 1 else 0

            # Aplicar filtros (a lógica não muda)
            c3_filtered = notch_filter(bandpass_filter(c3_signal, 0.5, 40, fs), 50, fs)
            c4_filtered = notch_filter(bandpass_filter(c4_signal, 0.5, 40, fs), 50, fs)
            
            # Processar condições (usando os IDs numéricos do MNE)
            conditions = event_map_mne.values() # [1, 2, 3]
            
            for cond_id in conditions:
                # Calcular STFT para cada condição e canal
                c3_epoch_data = compute_individual_epoch_stft(c3_filtered, events, cond_id, tmin=0, tmax=epoch_duration_sec, fs=fs,
                                                              nperseg=nperseg, noverlap=noverlap, nfft=nfft)
                c4_epoch_data = compute_individual_epoch_stft(c4_filtered, events, cond_id, tmin=0, tmax=epoch_duration_sec, fs=fs,
                                                              nperseg=nperseg, noverlap=noverlap, nfft=nfft)

                # Salvar os espectrogramas
                class_folder_name = event_map_folders[cond_id]
                class_output_dir = os.path.join(base_output_data_dir, class_folder_name)
                
                for j in range(len(c3_epoch_data)):
                    epoch_idx = c3_epoch_data[j]['epoch_idx']
                    Zxx_c3 = c3_epoch_data[j]['magnitude']
                    Zxx_c4 = c4_epoch_data[j]['magnitude']

                    base_filename = f"{subject_id}_{edf_file.stem.replace('.', '_')}_epoch_{epoch_idx+1}"
                    
                    np.save(os.path.join(class_output_dir, f"{base_filename}_C3.npy"), Zxx_c3)
                    np.save(os.path.join(class_output_dir, f"{base_filename}_C4.npy"), Zxx_c4)

print("\nProcessamento concluído.")

Iniciando a geração de spectrograms para sujeitos S001 a S085
e para as janelas: [200]ms

--- Processando com Janela de 200ms ---
Diretório de saída para o dataset unificado: unified_spectrograms_200ms
  Processando Sujeito: S001
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
  Processando Sujeito: S002
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
  Processando Sujeito: S003
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(